# Module 18 — Next-Token Prediction: The Core Objective

**Part V · Generation & Alignment · Module 1 of 5**

Welcome to Part V. We've spent four parts assembling the machine: tokenizers,
embeddings, attention, transformers, KV caches, MoE, quantization, LoRA. Now
we get to use it for what it was actually built to do — *predict the next
token*. That sentence sounds boring until you stare at it for long enough.
Then it starts to feel like the only thing in the world.

In Part III you trained nanoGPT and watched the loss go down. You know the
loss is "cross-entropy on next tokens." This module zooms in on what that
distribution actually *looks like* inside a real model. You'll feed prompts
into GPT-2, pull out the raw probabilities over all ~50k vocabulary entries,
and watch the model commit, hesitate, sharpen, and waver as you give it more
context.

By the end of this notebook you should have answers — *and a feeling* — for:

- What does $P(x_t \mid x_{<t})$ literally compute, in code?
- Why is perplexity a more humane number than cross-entropy loss?
- How "spiky" is a real model's output distribution? How long is the long tail?
- What happens to the distribution as you add one more word of context?
- What if we mess with the logits before softmax? (Spoiler: that's Module 19.)


## 0. Setup

Nothing exotic. We need `torch`, `transformers`, `numpy`, and `matplotlib`. GPT-2 small is ~500 MB; the first run will download it from HuggingFace.

In [ ]:
# If you're in Colab, uncomment:
# !pip install -q transformers torch matplotlib numpy

import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


In [ ]:
# Our house palette — same one we've been using since Part I.
PALETTE = {
    "ink":    "#1a1a2e",
    "paper":  "#f7f3e9",
    "rose":   "#e63946",
    "amber":  "#f4a261",
    "teal":   "#2a9d8f",
    "indigo": "#3d5a80",
    "plum":   "#7b2cbf",
}

mpl.rcParams.update({
    "figure.facecolor":  PALETTE["paper"],
    "axes.facecolor":    PALETTE["paper"],
    "savefig.facecolor": PALETTE["paper"],
    "axes.edgecolor":    PALETTE["ink"],
    "axes.labelcolor":   PALETTE["ink"],
    "xtick.color":       PALETTE["ink"],
    "ytick.color":       PALETTE["ink"],
    "text.color":        PALETTE["ink"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "DejaVu Sans",
    "axes.titleweight":  "bold",
    "figure.dpi":        110,
})


In [ ]:
# Load GPT-2 small. ~124M parameters, vocab size 50257.
MODEL_NAME = "gpt2"
tok = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device).eval()

V = model.config.vocab_size
H = model.config.n_embd
print(f"vocab size V = {V:,}")
print(f"hidden size d = {H}")
print(f"params      = {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


## 1. The one equation that runs the whole show

A language model is a function. You hand it a sequence of tokens
$x_1, x_2, \dots, x_{t-1}$ and it gives you back a probability distribution
over what comes next. That's it. That's the job.

Inside the transformer, by the time you reach the top of the stack, the
last position has been crushed down into a single vector $h_t \in \mathbb{R}^d$
— a $d$-dimensional summary of everything the model thinks about what should
happen next. To turn that vector into a distribution over the vocabulary, we
multiply it by a learned matrix $W_{\text{head}} \in \mathbb{R}^{V \times d}$
and softmax the result. Symbol-by-symbol:

- $x_{<t}$ — every token before position $t$ (the context).
- $h_t$ — the last hidden state, after $L$ transformer blocks. One vector, $d$ numbers.
- $W_{\text{head}}$ — the *unembedding* matrix. One row per vocabulary token. In GPT-2 it's literally tied to the input embedding matrix.
- $z_t = W_{\text{head}} \cdot h_t$ — the **logits**. A vector of length $V$. Unnormalized scores. Can be any real number.
- $\text{softmax}$ — the thing that turns scores into probabilities by exponentiating and normalizing.

$$
P(x_t \mid x_{<t}) \;=\; \text{softmax}(W_{\text{head}}\, h_t)
\;=\; \frac{\exp(z_{t,i})}{\sum_{j=1}^{V} \exp(z_{t,j})}
$$

The whole 124M-parameter circus exists to compute $h_t$. The last step —
$W_{\text{head}} \cdot h_t$ followed by softmax — is the same matrix multiply
and the same softmax that a logistic regression student would write in their
first ML class. The magic is everything that happens *before* it.


## 2. Computing $P(x_t \mid x_{<t})$ by hand

Let's not trust HuggingFace's `.generate()`. Let's pull out $h_t$ ourselves, multiply it by $W_{\text{head}}$ ourselves, softmax it ourselves, and compare to what the model says. If the numbers match, we *understand* it.

In [ ]:
prompt = "The cat sat on the"
input_ids = tok.encode(prompt, return_tensors="pt").to(device)
print("tokens:", [tok.decode([t]) for t in input_ids[0].tolist()])
print("shape :", tuple(input_ids.shape))


In [ ]:
# Run the transformer, asking it to also return the final hidden states.
with torch.no_grad():
    out = model(input_ids, output_hidden_states=True)

# out.hidden_states is a tuple of (L+1) tensors of shape (1, T, d)
# Index -1 is the output of the final transformer block, AFTER the final layernorm.
h_last = out.hidden_states[-1][0, -1]   # (d,)  -- the famous h_t
print("h_t shape:", h_last.shape)
print("h_t[:8] =", h_last[:8].cpu().numpy().round(3))


In [ ]:
# Manual unembedding: z = W_head @ h_t
W_head = model.lm_head.weight       # (V, d), tied to input embeddings in GPT-2
z_manual = W_head @ h_last          # (V,)
p_manual = F.softmax(z_manual, dim=-1)

# What the model itself computed, for comparison.
z_model = out.logits[0, -1]
p_model = F.softmax(z_model, dim=-1)

# Sanity check: these should be identical to floating-point precision.
print("max |Δ logits| :", (z_manual - z_model).abs().max().item())
print("max |Δ probs|  :", (p_manual - p_model).abs().max().item())
print("∑ p (should = 1):", p_manual.sum().item())


Two near-zero numbers and a one. We didn't just *use* the equation — we rebuilt the last step ourselves and matched the model bit-for-bit. From here on, the distribution `p_model` is the only thing that matters.

## 3. What does the model think comes after "The cat sat on the"?

Let's look at the top of the distribution. If the world is fair, "mat" should be in there somewhere — but how confident is the model, really?

In [ ]:
def top_k_table(probs, k=15):
    vals, idx = torch.topk(probs, k)
    return [(tok.decode([int(i)]), float(v)) for i, v in zip(idx, vals)]

table = top_k_table(p_model, k=15)
print(f"{'rank':>4}  {'token':>14}  {'P':>9}")
print("-" * 33)
for r, (t, p) in enumerate(table, 1):
    bar = "█" * int(round(p * 60))
    print(f"{r:>4}  {repr(t):>14}  {p:>8.3%}  {bar}")


So the model is *betting* on " floor" or " ground" — " mat" is in the top 15 but it's nowhere near 100%. There is no single right answer. Real text continues a thousand different ways, and the model has been trained to spread its belief proportionally. This is the most important thing to internalize about LLMs: **they are not lookup tables**. They are distributions.

## 4. The full distribution and the long tail

Top-15 hides the real story. The vocabulary has 50,257 tokens. Let's plot the *whole thing*, sorted from most to least likely, on a log scale. Heads up: this is the kind of plot that teaches you a fact you can't un-learn.

In [ ]:
probs_sorted, _ = torch.sort(p_model, descending=True)
probs_np = probs_sorted.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) linear, top 50 only -- shows the spike
ax = axes[0]
top = 50
ax.bar(range(top), probs_np[:top], color=PALETTE["plum"], edgecolor=PALETTE["ink"], linewidth=0.5)
ax.set_title("Top 50 tokens, linear scale")
ax.set_xlabel("rank")
ax.set_ylabel("P(token)")
ax.set_xlim(-0.5, top - 0.5)

# (b) log scale, all 50,257 tokens -- shows the long tail
ax = axes[1]
ax.plot(probs_np, color=PALETTE["rose"], lw=1.2)
ax.set_yscale("log")
ax.set_title(f"All {V:,} tokens, log scale")
ax.set_xlabel("rank")
ax.set_ylabel("P(token)  (log)")
ax.axhline(1e-6, color=PALETTE["ink"], ls=":", lw=0.8, alpha=0.5)
ax.text(V * 0.55, 1.5e-6, "1 in a million", color=PALETTE["ink"], fontsize=9, alpha=0.7)

fig.suptitle('"The cat sat on the ___" -- distribution over the next token',
             fontsize=13, color=PALETTE["ink"])
plt.tight_layout()
plt.show()


Look at that right-hand plot. The first hundred tokens carry almost all the mass, but there are still tens of thousands of tokens with **strictly positive probability**. Softmax never outputs zero. The model assigns non-trivial probability to " pizza", " president", " zebra", " }};". Most of these will never be sampled in practice — but they exist in the distribution, and they are the reason temperature, top-k, and top-p will matter so much in Module 19.

In [ ]:
# How much mass is in the top-k for various k?
cum = torch.cumsum(probs_sorted, dim=0).cpu().numpy()
for k in [1, 5, 10, 50, 100, 500, 1000, 5000]:
    print(f"top-{k:>5}: {cum[k-1]:.4f}  of probability mass")


Top-1 alone holds maybe ~15% of belief. Top-50 holds something like 85–95%. The remaining ~50,000 tokens together carry the last few percent. That's the long tail. It's small in mass and gigantic in vocabulary count.

## 5. From the distribution to the loss (and to perplexity)

Training a language model is one rule applied a trillion times: *make the
true next token as likely as possible*. If the true token at position $t$ is
$x_t$ and the model put probability $P(x_t \mid x_{<t})$ on it, the loss for
that one position is the negative log probability:

$$
\ell_t \;=\; -\log P(x_t \mid x_{<t})
$$

Plain English: if the model said the truth had probability 1, the loss is 0.
If the model said 0.5, the loss is $\log 2 \approx 0.69$. If the model said
$10^{-9}$, the loss is about 21. Big penalty for being confidently wrong.

Average that over a sequence and you get the cross-entropy loss the trainer
prints:

$$
\mathcal{L} \;=\; -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t})
$$

Now here's the move that makes loss numbers *interpretable*. Exponentiate it:

$$
\text{PPL} \;=\; e^{\mathcal{L}}
$$

Why bother? Because perplexity is the **effective vocabulary size** the model
is choosing from. PPL = 1 means the model knew the answer with certainty
every single step. PPL = $V$ means it was picking uniformly at random across
the entire vocabulary — totally clueless. PPL = 50 means "at every step, the
model is effectively rolling a 50-sided die." That's a number a human can
hold in their head; a loss of 3.91 is not.


In [ ]:
def loss_and_ppl(text):
    ids = tok.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(ids).logits  # (1, T, V)
    # Shift so we predict ids[t] from logits at position t-1
    shift_logits = logits[0, :-1, :]      # (T-1, V)
    shift_targets = ids[0, 1:]             # (T-1,)
    log_probs = F.log_softmax(shift_logits, dim=-1)
    nll = -log_probs.gather(1, shift_targets.unsqueeze(1)).squeeze(1)  # (T-1,)
    L = nll.mean().item()
    return L, math.exp(L), nll.cpu().numpy()

samples = [
    "The cat sat on the mat.",
    "The capital of France is Paris.",
    "Quantum chromodynamics describes the strong interaction.",
    "asdf qwerty zxcv lkj poiu mnbv",                # gibberish
    "The the the the the the the the the the the.", # repetitive
]

print(f"{'L':>7}  {'PPL':>9}   text")
print("-" * 70)
for s in samples:
    L, ppl, _ = loss_and_ppl(s)
    print(f"{L:>7.3f}  {ppl:>9.2f}   {s!r}")


Read those PPLs as "effective number of choices per step." Real English sentences sit somewhere between ~10 and ~80. Gibberish tokens explode into the thousands — at every step the model is genuinely surprised. The repetition example is fascinating: once the pattern locks in, the model gets *very* confident the next token is also "the", and the PPL collapses. The model has learned that boring text is easy text.

## 6. Where, exactly, is the model surprised?

The mean loss tells you the average. But losses *within* a sentence vary wildly. Let's plot the per-token surprisal $-\log P(x_t)$ for one sentence and see which tokens hurt and which were obvious.

In [ ]:
sentence = "The Eiffel Tower is located in the city of Paris, France."
ids = tok.encode(sentence, return_tensors="pt").to(device)
toks = [tok.decode([i]) for i in ids[0].tolist()]
L, ppl, nll = loss_and_ppl(sentence)

fig, ax = plt.subplots(figsize=(12, 4))
xs = np.arange(len(nll))
colors = [PALETTE["rose"] if v > 4 else (PALETTE["amber"] if v > 1.5 else PALETTE["teal"]) for v in nll]
ax.bar(xs, nll, color=colors, edgecolor=PALETTE["ink"], linewidth=0.5)
ax.set_xticks(xs)
ax.set_xticklabels([t.strip() or "·" for t in toks[1:]], rotation=45, ha="right", fontsize=10)
ax.set_ylabel(r"$-\log P(x_t \mid x_{<t})$  (nats)")
ax.set_title(f'Per-token surprisal · mean loss = {L:.2f}, PPL = {ppl:.1f}')
ax.axhline(L, color=PALETTE["ink"], ls=":", lw=1.2, label=f"mean = {L:.2f}")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("\nFor each predicted token: how surprised was the model?")
for t, v in zip(toks[1:], nll):
    print(f"  {repr(t):>14}  {v:6.3f} nats   {'■' * int(round(v * 4))}")


"Eiffel" is cheap once you've seen "The". "Tower" is essentially free once you've seen "Eiffel". But "Paris" — the actual fact — costs more. The model has to remember a piece of world knowledge to put high probability on it. Cross-entropy loss isn't an abstract scalar; it's a vector of *moments where the model had to think*.

## 7. The distribution sharpens as you give more context

Here's the experiment that gave me a feeling for what context *does* to an LLM. We feed in "The" and look at the next-token distribution. Then "The cat". Then "The cat sat". Then "The cat sat on". Then "The cat sat on the". With each new word the model has more to go on, and its uncertainty (entropy) about the next token should drop.

Entropy of a distribution is the average surprisal *the model expects for itself*:

$$ H(P) \;=\; -\sum_{i=1}^{V} P(x_i) \log P(x_i) $$

It's measured in nats here (we're using natural log). $H = 0$ means "I am 100% sure". $H = \log V$ means "I have no idea, every token is equally likely." For GPT-2's vocab, that maximum is about $\log 50257 \approx 10.8$ nats.

In [ ]:
def entropy(p, eps=1e-12):
    p = p.clamp_min(eps)
    return -(p * p.log()).sum().item()

def next_token_distribution(text):
    ids = tok.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    return F.softmax(logits, dim=-1)

stages = [
    "The",
    "The cat",
    "The cat sat",
    "The cat sat on",
    "The cat sat on the",
]

dists = [next_token_distribution(s) for s in stages]
entropies = [entropy(p) for p in dists]
top1 = [float(p.max()) for p in dists]
H_max = math.log(V)

for s, H, t1 in zip(stages, entropies, top1):
    print(f"H = {H:5.2f} nats   top-1 P = {t1:5.1%}   '{s} ___'")
print(f"\n(maximum possible entropy = log V = {H_max:.2f} nats)")


In [ ]:
# Plot 1: entropy vs. context length
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
xs = np.arange(len(stages))
ax.plot(xs, entropies, "o-", color=PALETTE["indigo"], lw=2, ms=10,
        markerfacecolor=PALETTE["paper"], markeredgewidth=2)
ax.set_xticks(xs)
ax.set_xticklabels([f'"{s}"' for s in stages], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("entropy (nats)")
ax.set_title("Uncertainty drops as context grows")
ax.axhline(H_max, color=PALETTE["rose"], ls=":", lw=1, label=f"max = log V = {H_max:.2f}")
ax.legend(loc="upper right", fontsize=9)

# Plot 2: top-1 probability climbing
ax = axes[1]
ax.plot(xs, [t * 100 for t in top1], "s-", color=PALETTE["teal"], lw=2, ms=10,
        markerfacecolor=PALETTE["paper"], markeredgewidth=2)
ax.set_xticks(xs)
ax.set_xticklabels([f'"{s}"' for s in stages], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("P(top token)  %")
ax.set_title("The model commits harder")

plt.tight_layout()
plt.show()


In [ ]:
# What is the top-1 token at each stage? It changes!
for s, p in zip(stages, dists):
    top = top_k_table(p, k=3)
    triple = "  ".join([f"{repr(t):>10} {v:.2%}" for t, v in top])
    print(f'"{s} ___"  ->  {triple}')


This is the bit that makes the abstract "language model" idea click. After "The", the model is genuinely lost — entropy is up around 8 nats, and its top guess (probably " first" or " most") is just a few percent. Add "cat" and the world contracts. Add "sat on the" and the model starts placing real bets on physical surfaces. Each context token is a constraint that knocks out enormous fractions of the vocabulary.

## 8. Watching the top of the distribution sharpen

One more view of the same phenomenon: the *shape* of the top-15 bars as we add context.

In [ ]:
fig, axes = plt.subplots(1, len(stages), figsize=(16, 4.2), sharey=True)
for ax, s, p in zip(axes, stages, dists):
    top = top_k_table(p, k=10)
    labels = [t for t, _ in top]
    vals = [v for _, v in top]
    ax.bar(range(len(vals)), vals, color=PALETTE["plum"],
           edgecolor=PALETTE["ink"], linewidth=0.5)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.set_title(f'"{s}"', fontsize=10)
axes[0].set_ylabel("P(token)")
plt.suptitle("Top-10 sharpening as context grows", fontsize=13)
plt.tight_layout()
plt.show()


From a flat little hedge of guesses to a clean spike on a single physical surface. *That's what context does.*

## 9. Break things on purpose: rescaling the logits

The softmax is sensitive to the *scale* of its inputs, not just their order.
That gives us a free knob: what happens if, before softmax, we multiply the
logits by some scalar $\alpha$?

$$
P_\alpha(x_i) \;=\; \frac{\exp(\alpha\, z_i)}{\sum_j \exp(\alpha\, z_j)}
$$

- $\alpha > 1$: gaps between logits get exaggerated. The biggest one wins by more. The distribution **sharpens**.
- $\alpha < 1$: gaps shrink. Everything looks closer to uniform. The distribution **flattens**.
- $\alpha \to \infty$: argmax. (Greedy decoding.)
- $\alpha \to 0$: uniform.

You may have already guessed: this is *exactly* temperature, with $\alpha =
1/T$. Module 19 will turn this into a proper sampling story. For now let's
just see what it does to perplexity. Cranking the temperature changes how
much probability the true tokens get — flattening the distribution should
make the right answers harder to find (PPL up); sharpening it should make
the right answers easier (PPL down)... right? Let's find out.


In [ ]:
def ppl_with_alpha(text, alpha):
    ids = tok.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(ids).logits  # (1, T, V)
    shift_logits = logits[0, :-1, :] * alpha
    shift_targets = ids[0, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    nll = -log_probs.gather(1, shift_targets.unsqueeze(1)).squeeze(1)
    return math.exp(nll.mean().item())

text = ("The Eiffel Tower is located in the city of Paris, France. "
        "It was built between 1887 and 1889 and stands 330 meters tall.")

alphas = np.array([0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0])
ppls = [ppl_with_alpha(text, a) for a in alphas]

for a, p in zip(alphas, ppls):
    print(f"alpha = {a:5.2f}  (T = {1/a:5.2f})  ->  PPL = {p:9.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alphas, ppls, "o-", color=PALETTE["rose"], lw=2, ms=9,
        markerfacecolor=PALETTE["paper"], markeredgewidth=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"logit scale $lpha$  (= 1 / temperature)")
ax.set_ylabel("Perplexity on the held-out sentence")
ax.set_title("Flattening the logits hurts. Sharpening them helps... up to a point.")
ax.axvline(1.0, color=PALETTE["ink"], ls=":", lw=1, alpha=0.6)
ax.text(1.05, ppls[0] * 0.6, "α = 1
(the trained model)", fontsize=9, color=PALETTE["ink"])

# Annotate the extremes
ax.annotate("flatten
(temperature ↑)", xy=(alphas[0], ppls[0]),
            xytext=(0.13, ppls[0] * 0.35),
            arrowprops=dict(arrowstyle="->", color=PALETTE["plum"]),
            fontsize=10, color=PALETTE["plum"])
ax.annotate("sharpen
(temperature ↓)", xy=(alphas[-1], ppls[-1]),
            xytext=(2.5, ppls[-1] * 4.5),
            arrowprops=dict(arrowstyle="->", color=PALETTE["teal"]),
            fontsize=10, color=PALETTE["teal"])
plt.tight_layout()
plt.show()


Two things to notice.

**One:** at $\alpha = 0.1$ (i.e. temperature 10), the distribution is
basically uniform, the PPL skyrockets, and the model is no better than a
random typist. We've thrown away everything the model knows.

**Two:** at $\alpha = 10$ (temperature 0.1), the PPL on this sentence
actually keeps going *down*. That looks like cheating, but it makes sense:
this is a coherent factual sentence where, conditional on the prefix, the
"correct" next token is very often *also* the highest-logit token. Sharpening
means committing harder to your top guess, and on easy text your top guess is
right. Try it on the gibberish or repetition strings and you'll see the
opposite — sharpening commits harder to the *wrong* answer.

**The lesson hiding under the surface:** the same model, the same hidden
states, can be made to behave like a confident expert or a lost gambler just
by rescaling its logits. We have not changed a single weight. We've changed
how we *read* its output. That little observation is the entire foundation
of Module 19.


In [ ]:
# Quick demo of the gibberish case to drive the point home.
gib = "asdf qwerty zxcv lkj poiu mnbv"
print(f"{'alpha':>6}  {'PPL (English)':>16}  {'PPL (gibberish)':>18}")
for a in [0.5, 1.0, 2.0, 5.0]:
    eng = ppl_with_alpha(text, a)
    g   = ppl_with_alpha(gib,  a)
    print(f"{a:>6.2f}  {eng:>16.2f}  {g:>18.2f}")


On English, sharpening helps. On gibberish, sharpening *hurts more*, because there's no top guess worth committing to — every position is genuinely uncertain, and forcing the distribution to a spike just concentrates the loss on whatever wrong guess happened to win.

## 10. Checkpoint quiz

Five questions. Try to answer before peeking at the next code cell.

1. The output head's matrix $W_{\text{head}}$ has shape $(V, d)$. In GPT-2,
   what other matrix in the model has *exactly the same shape* and (in fact)
   the same values?

2. A model gets a mean cross-entropy loss of $\ln 50 \approx 3.91$ nats on
   a test set. What is the perplexity, and what does that number mean
   intuitively?

3. You feed GPT-2 the prefix "The cat sat on the". Roughly what fraction of
   the 50,257 vocabulary tokens get *exactly zero* probability?

4. As you add more context tokens to a prompt, what should happen to the
   entropy of the next-token distribution, and why? Give one sentence each.

5. We multiplied the logits by $\alpha$ before softmax. With $\alpha = 0$,
   what does the resulting distribution look like? With $\alpha \to \infty$,
   what does it converge to?


### Answers

1. The **input embedding matrix** `wte`. GPT-2 (and most modern LMs) *tie*
   the input embeddings and the output projection so they share parameters.
   You can verify it: `model.lm_head.weight is model.transformer.wte.weight`
   evaluates to `True` for GPT-2. This halves the parameter count of the
   output layer and improves training.

2. PPL = $e^{3.91} = 50$. Intuitively: at every position, the model is as
   uncertain as if it were rolling a fair 50-sided die. Average loss in nats
   is the log of effective vocabulary size; perplexity undoes the log so you
   get a number you can hold in your head.

3. **Zero.** Softmax of a finite real-valued vector is strictly positive
   everywhere — every token in the vocabulary, including " zebra" and "}};",
   has nonzero probability. That's the long tail you saw on the log-scale
   plot. It's small in mass, but it never collapses to zero, which is why
   sampling strategies in Module 19 have to *manually* truncate it.

4. Entropy should generally **drop**. More context means more constraints
   on what's plausible next, so the model can concentrate its probability
   mass on fewer tokens. (It can occasionally *rise* at sentence boundaries
   or topic shifts, but the overall trend is down.)

5. With $\alpha = 0$, every logit becomes $0$, so $\exp(0) = 1$ everywhere,
   and softmax produces the **uniform distribution** over $V$ tokens. With
   $\alpha \to \infty$, the gap between the largest logit and everything
   else gets infinitely amplified, so all the mass concentrates on the
   argmax — that's **greedy decoding**, the deterministic limit.


## Bridge to Module 19

We now have two solid facts about the inside of an LLM:

- The model's output is a **full distribution** over the vocabulary, and
  it's *never* a delta function. There is always a tail.
- We can re-shape that distribution after the fact — sharpen it, flatten it,
  truncate it — without touching a single weight.

So the obvious next question is: given this distribution, *how do we pick
one token from it?* Argmax (greedy)? Sample proportionally? Top-k? Top-p?
Min-p? Each choice gives wildly different generations from *the exact same
model*. The same prompt that produces a Wikipedia-style answer at
temperature 0.2 produces a fever dream at temperature 1.5.

That's Module 19. Bring snacks.
